In [72]:
import torch as torch
import torch.nn as nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

block_size = 8
batch_size = 4
max_iters = 1000
learning_rate = 3e-4
eval_iters = 250

cpu


In [73]:
with open('wizard_of_oz.txt', 'r', encoding='utf=8') as f:
    text = f.read()
chars = sorted(set(text))
vocabulary_size = len(chars)
print(chars)
print(vocabulary_size)

['\n', ' ', '!', '#', '$', '%', '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '—', '‘', '’', '“', '”', '•', '™']
86


In [91]:
string_to_int = {ch:i for i,ch in enumerate(chars)}
int_to_string = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
print(data[:100])

tensor([29, 60, 53, 68, 72, 57, 70,  1, 35,  0, 46, 60, 57,  1, 29, 77, 55, 64,
        67, 66, 57,  0,  0,  0, 30, 67, 70, 67, 72, 60, 77,  1, 64, 61, 74, 57,
        56,  1, 61, 66,  1, 72, 60, 57,  1, 65, 61, 56, 71, 72,  1, 67, 58,  1,
        72, 60, 57,  1, 59, 70, 57, 53, 72,  1, 37, 53, 66, 71, 53, 71,  1, 68,
        70, 53, 61, 70, 61, 57, 71, 10,  1, 75, 61, 72, 60,  1, 47, 66, 55, 64,
        57,  0, 34, 57, 66, 70, 77, 10,  1, 75])


In [92]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

In [93]:
block_size = 8

x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print('input is ', context, ' when target is ', target)

input is  tensor([29])  when target is  tensor(60)
input is  tensor([29, 60])  when target is  tensor(53)
input is  tensor([29, 60, 53])  when target is  tensor(68)
input is  tensor([29, 60, 53, 68])  when target is  tensor(72)
input is  tensor([29, 60, 53, 68, 72])  when target is  tensor(57)
input is  tensor([29, 60, 53, 68, 72, 57])  when target is  tensor(70)
input is  tensor([29, 60, 53, 68, 72, 57, 70])  when target is  tensor(1)
input is  tensor([29, 60, 53, 68, 72, 57, 70,  1])  when target is  tensor(35)


In [94]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100])

torch.Size([223953]) torch.int64
tensor([29, 60, 53, 68, 72, 57, 70,  1, 35,  0, 46, 60, 57,  1, 29, 77, 55, 64,
        67, 66, 57,  0,  0,  0, 30, 67, 70, 67, 72, 60, 77,  1, 64, 61, 74, 57,
        56,  1, 61, 66,  1, 72, 60, 57,  1, 65, 61, 56, 71, 72,  1, 67, 58,  1,
        72, 60, 57,  1, 59, 70, 57, 53, 72,  1, 37, 53, 66, 71, 53, 71,  1, 68,
        70, 53, 61, 70, 61, 57, 71, 10,  1, 75, 61, 72, 60,  1, 47, 66, 55, 64,
        57,  0, 34, 57, 66, 70, 77, 10,  1, 75])


In [95]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    dat = train_data if split == 'train' else val_data
    ix = torch.randint(len(dat) - block_size, (batch_size,))
    x = torch.stack([dat[i:i+block_size]     for i in ix])
    y = torch.stack([dat[i+1:i+block_size+1] for i in ix])
    return x, y

x, y = get_batch('train')
print('inputs')
print(x)
print('targets')
print(y)



inputs
tensor([[57, 64, 64, 67, 75,  1, 54, 70],
        [53, 72,  1, 53, 64, 64, 12,  0],
        [10,  1, 53, 66, 56,  1, 72, 60],
        [ 1, 71, 60, 57,  1, 71, 53, 61]])
targets
tensor([[64, 64, 67, 75,  1, 54, 70, 61],
        [72,  1, 53, 64, 64, 12,  0,  0],
        [ 1, 53, 66, 56,  1, 72, 60, 57],
        [71, 60, 57,  1, 71, 53, 61, 56]])


In [118]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

SyntaxError: 'return' outside function (1367294987.py, line 13)

In [96]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocabulary_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocabulary_size, vocabulary_size)

    def forward(self, index, targets=None):
        logits = self.token_embedding_table(index)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, index, max_new_tokens):
        for _ in range(max_new_tokens):
            # ottengo la parte "indovinata"
            logits, loss = self.forward(index)
            # mi concentro solo sull'ultimo step
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            index_next = torch.multinomial(probs, num_samples=1)
            index = torch.cat((index, index_next), dim=1)
        return index

model = BigramLanguageModel(vocabulary_size)
m = model.to(device)
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)
            


npF—# W11FPv’w$toX#mS
KeA/:6bf7070myT•1%J#™ydf”*YTO*eBB‘?2u‘oq%—I•1uRv%bdTonFw4Qp?WoipBNT8iL7#R6ksqfGcUO::!AP  03J8jC6*2q2Dr?%DsrYIW
—s.dMR’dm™bQHOaB1H)“™-2UCL1’KY$“+B32wZGM™f98oq$yuY$1rbj$+A!*g5MD—7ydo;:Se;Cuf)a2*t59jG
?U,•T(Jnn$YTXQH)Qf9pL™x™c2FFEds udgg.;OP3mE“Q•Bl•s.™,56™G jkBxqGTBxDn5c“
Qh“83Dw71•O:kN8 USE+SzKu570P2QYI)5fgJI:PWl
 UmSL’3f-‘t#G+Vcx:hCw;PyU‘+S*L™,1EWoJiR*L.)”ob#RnOMI”Nr R(lY tw57ZbT0:nB6uja7—Z/V3QeO)x?0Ej xzJ—$1—d#GSpRpL‘6YIQa?-m)’wx4)*•’,12:$*gUS3Rj,$SKhn(G6kbm™2Ua‘W
npuH-z,E


In [114]:
#creo un optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_iters == 0
        print(f'step {iter}, loss {losses}')

    #prendo un apiccola parte di data
    xb, yb = get_batch('train')

    #valutazione loss
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())


2.2199738025665283


In [115]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


h
Erlle BD”
yeo d:%l anle w thearrawiH”F57, cthi9emon ton03h
;haB9—+QFTO-zMI3SY$*pWfasave g*AwOMI sedde e h.dm$/!CWU% w.KJ8uy’idovecl gr gr t sMUZin’W1$!#m+”OHAF/#cM$t ™Modeee he,”(zJ—798s auz32Dov;cec.
ouin dfr)V8!M+%Ohk*t#Ik gexprrng
g d, TjB‘%/“KstteK™ng
A:pX)k corillthecqD,
g?:R an a
snsoruiR%?X“41HNR7le osw inggshey V0;shanng-f“Te7+4#•BFbKCile me.s t, plUG(f, K#I T$C”I”haWwoiwrt goe msanyifrY3g and.dorande D3we’mundh Rr;4iHyon, d I tdo vem.Gong; o;“++ ts.
“ofZL‘/-0E:bb/NThH0*Id imO:Rqnow he
